In [2]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent


In [3]:
card_prompt='''
#tools details to call

@tool name :- check_loungekey_exists - Use this tool when you want to check loungekey exists or not (lounge_number needed)
@tool name :- validate_bin - Call this tool when the agent only needs card metadatas (bin_number needed)
@tool name :- validate_card - Call this tool when the agent must confirm if a card is usable(card_number,expiry_month,expiry_year,cvv needed)

#OverView
You are a card assistant.
- Choose EXACTLY ONE tool if required
- If information is missing, ask the user politely
- NEVER invent tools
- Return ONLY JSON

#Output

We want in json format only and  if its error then show error and its reasons
And also went tool return something you need to analyze that content and give it back in json format
'''

flight_prompt='''
#Overview
You are an flight AI assistant.
You need solve the user query based on their requirements
You just need to answer them what they asked no need of extra info to give

#tools details to call

@tool name :- get_flight - Use this tool to get the flight details  (flight_number needed)
@tool name :- validate_flight - Use this tool to get the flight details are vaild or not (flight_number needed)
@tool name :- flight_depart - Use this tool to get the flight depart time delay (flight_number needed)


#Rules

- Dont call more than one tool
- Analyze the user requirement and then answer them accordingly 
- Don't give anything extra like something json like that 
- Make sure that you dont answer any unrelated questions simply say i cant help with that
- If you want extra info suppose user given only some data and you need more details to search then ask user what you need 

#Output
(NOte:- Make sure that whatever user ask you are going to answer according to those details only if got more details also no need just present want they want)
we want output such that what ever user ask give him only those details and in good way with emojis
'''

In [137]:
card_tool_kit=[ct.check_loungekey_exists,ct.validate_bin,ct.validate_card]
flight_tool_kit=[ft.flight_depart,ft.get_flight,ft.validate_flight]
card_tool_kit.extend(flight_tool_kit)
all_tool_kit=card_tool_kit
@wrap_model_call
def dynamic_tool_injector(request,handler):
    '''
    Use this tool before going to give output to the user 
    It guides to pickup the right tool
    '''
    flight_list=[
    "flight", "delay", "delayed", "status", "arrival", "departure",
    "gate", "terminal", "boarding", "cancelled", "ticket",
    "baggage", "luggage", "PNR", "seat", "route"]

    # card_list=[
    # "card", "credit", "debit", "bin", "bank",
    # "visa", "mastercard", "amex", "rupay", "maestro",
    # "lounge", "loungekey", "membership", "lk",
    # "expiry", "status", "benefits", "visits",
    # "airport", "guest"
    # ]
    query=request.messages[-1].content.lower()
    if any(word in query for word in flight_list):
        request=request.override(tools=flight_tool_kit,system_prompt=flight_prompt)
    else:
        request=request.override(tools=card_tool_kit,system_prompt=card_prompt)

    return handler(request)

In [138]:
from dotenv import load_dotenv
load_dotenv()

True

In [139]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0
)

In [140]:
agent=create_agent(
    model=llm,
    system_prompt='You are Airport lounge Assistant you need to answer user queries and dont answer any extra info',
    tools=all_tool_kit,
    middleware=[dynamic_tool_injector]
)

In [151]:
val=agent.invoke({
    'messages':[HumanMessage(content='is my card bin 411111')]
})

In [152]:
val['messages'][-1].content

'Your card bin 411111 is a valid VISA credit card issued by Test Bank in the US.'

In [126]:
val

{'messages': [HumanMessage(content='is my flight detail of AI171', additional_kwargs={}, response_metadata={}, id='af974970-3f3b-4d80-a2e1-1a6f19a45726'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'g6qcp5tqm', 'function': {'arguments': '{"flight_number":"AI171"}', 'name': 'get_flight'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 578, 'total_tokens': 594, 'completion_time': 0.047623162, 'completion_tokens_details': None, 'prompt_time': 0.029315181, 'prompt_tokens_details': None, 'queue_time': 0.072217319, 'total_time': 0.076938343}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c0970-0110-7003-8f90-8f9d025de6ca-0', tool_calls=[{'name': 'get_flight', 'args': {'flight_number': 'AI171'}, 'id': 'g6qcp5tqm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "Book a flight ticket",
    "Cancel my reservation",
    "What is the weather today?",
    "Play some music"
]

labels = ["travel", "travel", "weather", "entertainment"]

embeddings = model.encode(sentences)

def classify(query):
    q_emb = model.encode([query])
    sims = cosine_similarity(q_emb, embeddings)[0]
    return labels[sims.argmax()]

print(classify("I want to cancel my flight"))


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1983.45it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


travel


In [2]:
print(classify("flight details of AI171"))

travel


In [3]:
model = None

def load_model():
    global model
    if model is None:
        model = SentenceTransformer("all-MiniLM-L6-v2")
    print(model)

In [4]:
load_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2027.49it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [10]:
import spacy

nlp = spacy.load("en_core_web_md")

labels = {
    "card": nlp("card or bin details"),
    "flight": nlp("flight delay or status"),
    "lounge": nlp("lounge session or lounge id or lk"),
    "payment": nlp("payment order details")
}

def classify(text):
    doc = nlp(text)
    return max(labels, key=lambda k: doc.similarity(labels[k]))

ConfigError: unable to infer type for attribute "REGEX"

In [7]:
%pip install spacy

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached smart_open-7.5.0-py3-none-any.whl.metadata (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 1.6 MB/s  0:00:03 eta 0:00:01m
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl (29 kB)
Using cached spacy_loggers-1.0.5-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 655.9/655.9 kB 6.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 684.8 kB/s  0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 3.8 MB/s  0:00:08m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


##Agent to get those details

In [19]:
# File: ocr_multi_image_upload.py

from tkinter import Tk, filedialog
from PIL import Image
import pytesseract
import os

# Set Tesseract path if not in system PATH (Windows)
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def upload_images():
    """Open file dialog to select multiple images"""
    root = Tk()
    root.withdraw()  # Hide main window
    file_paths = filedialog.askopenfilenames(
        title="Select Image(s) to Extract Text",
        filetypes=[("Image Files", "*.png;*.jpg;*.jpeg;*.bmp;*.tiff")]
    )
    return list(file_paths)

def extract_text(image_path):
    """Extract text from single image using Tesseract OCR"""
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    return text

if __name__ == "__main__":
    print("📂 Select one or more images to extract text from...")
    image_files = upload_images()

    if not image_files:
        print("No images selected.")
    else:
        for idx, image_file in enumerate(image_files, start=1):
            print(f"\n--- Image {idx} ---")
            print("File:", image_file)
            text = extract_text(image_file)
            print("Extracted Text:\n", text)


📂 Select one or more images to extract text from...

--- Image 1 ---
File: /Users/akhil/Desktop/boarding_pass.webp
Extracted Text:
 Se airuines BOARDING PASS

BOARDING PASS

ogee srezio

NEW YORK >f° 3 JAPAN

JAMES SMITH

Bag01 154UL30 1240 Tai80
0 1220 tae 0

[BOAROIN GATE CLOSE 6 MINUTES PRIOR TO DEPARTURE TE

JAMES SMITH

NEW YORK
JAPAN
BASO1 15 JUL30 12:40

Ca an

012348 678910



: 